In [1]:
import sys
import numpy as np
import plotly.graph_objects as go

sys.path.insert(0, "/home2/s4636708/master_thesis_project/src")
from galaxy_sidm.io import load_config
from galaxy_sidm.mock import load_galaxy_gas

In [8]:
MODEL = "CDM"        # CDM, SIDM1 or vSIDM
REDSHIFT = 2         # 5, 4, 3, 2, 1 or 0.5
SUB_ID = 19743       # subhalo id
R_MAX = 30           # show particles within this radius [kpc]
MAX_POINTS = 500000   # at most this many stars and this many gas cells (random subset)

In [9]:
cfg = load_config()
snap = {float(z): int(s) for s, z in cfg["snap_z"].items()}[REDSHIFT]
run = {"CDM": "L35n1080_CDM", "SIDM1": "L35n1080_SIDM1", "vSIDM": "L35n1080_vSIDM_correa"}[MODEL]
gal = load_galaxy_gas(f"{cfg['paths']['aida_root']}/{run}/output/", snap, SUB_ID)

stars = gal.xyz_s.to_value("kpc")            # positions relative to the subhalo centre
gas = gal.xyz_g.to_value("kpc")
neutral = (gal.mH_neutral_g / gal.mgas_g).value  # neutral hydrogen (HI+H2) / gas mass, per cell


def pick(xyz):
    """Indices of the particles within R_MAX, at most MAX_POINTS of them."""
    idx = np.flatnonzero(np.linalg.norm(xyz, axis=1) < R_MAX)
    if len(idx) > MAX_POINTS:
        idx = np.random.default_rng(0).choice(idx, MAX_POINTS, replace=False)
    return idx


s, g = pick(stars), pick(gas)
print(f"{len(stars)} stars, {len(gas)} gas cells; plotting {len(s)} stars and {len(g)} gas cells")

fig = go.Figure([
    go.Scatter3d(x=stars[s, 0], y=stars[s, 1], z=stars[s, 2], mode="markers", name="stars",
                 marker=dict(size=1.5, color="orange", opacity=0.4)),
    go.Scatter3d(x=gas[g, 0], y=gas[g, 1], z=gas[g, 2], mode="markers", name="gas",
                 marker=dict(size=1.5, color=neutral[g], colorscale="Viridis", cmin=0, cmax=0.76,
                             opacity=0.4, colorbar=dict(title="HI+H2 /<br>gas mass"))),
])
fig.update_layout(title=f"{MODEL}  z={REDSHIFT}  subhalo {SUB_ID}  (click the legend to hide stars or gas)",
                  scene=dict(xaxis_title="x [kpc]", yaxis_title="y [kpc]", zaxis_title="z [kpc]",
                             aspectmode="cube"),
                  height=800)
fig.show()

112212 stars, 166102 gas cells; plotting 109752 stars and 53187 gas cells
